# 第 6 章 · 会话、状态与记忆：按读取方式、共享范围与存储后端选型

> **你已会**：Agent、工具、`output_key`、事件流。
>
> **读完后你能**：① 说清 Session / State / Memory / Artifact 各存什么；② 用 `user:` 前缀实现跨会话偏好；③ 用 SQLite 让会话活过进程重启；④ 用 Artifact 管理带版本的文件；⑤ 说清状态何时真正提交、会话如何列出与删除。
>
> **本章不碰**：跨进程恢复的执行细节（第 9 章）、云端存储。

**怎么读**

- **主线（约 50 分钟）**：1 Session → 2 State 作用域 → 3 Memory → 5 选型。
- **深读**：1.1 SQLite 持久化、4 Artifact 版本管理。
- 先回答每节的「先猜」，再运行。

先分清三个问题：**怎么取数据、谁能共享、进程退出后还在不在**。名称本身不保证保存时长：

| 构件 | 读取方式 | 一个具体例子 | 保存时长由谁决定 |
|---|---|---|---|
| Session events | 按事件历史读取 | 上一轮用户说了什么 | SessionService 后端 |
| State | 按键读取 | 用户等级、当前草稿 | SessionService 后端；前缀控制共享范围 |
| Memory | 按查询检索 | 从旧对话召回过敏记录 | MemoryService 后端 |
| Artifact | 按文件名与版本读取 | 已生成报告的第 1 版 | ArtifactService 后端 |

```mermaid
flowchart TB
    subgraph RUN["一次 Agent 运行"]
        EV["Event 事件流"]
        SD["state_delta 状态增量"]
    end
    subgraph SS["SessionService"]
        S1["Session: app + user + session_id<br/>├── events: 对话历史<br/>└── state: 键值状态"]
    end
    subgraph MS["MemoryService"]
        M1["跨会话可检索的记忆库"]
    end
    RUN --> SS
    SS -->|"add_session_to_memory"| MS
    MS -.->|"search_memory 召回"| RUN
    style S1 fill:#e8f0fe,stroke:#4285f4,stroke-width:2px
    style M1 fill:#e6f4ea,stroke:#34a853,stroke-width:2px
```

---

## 1. Session：一次对话的完整档案

`Session` 由 `app_name + user_id + session_id` 三元组定位，内含：

- `events`：从第一句"你好"开始的所有事件；
- `state`：伴随会话演化的键值状态；
- `last_update_time` 等元数据。

`SessionService` 是可插拔的。本章只涉及两种：

| 实现 | 数据活到什么时候 | 适合 |
|---|---|---|
| `InMemorySessionService` | 进程退出即丢 | 学习、单元测试 |
| `DatabaseSessionService` | 写进数据库，重启还在 | 本地持久化、跨进程恢复 |

> 💡 第 3 章已经验证了 State 提交。本章分别扩展**共享范围、后端和读取方式**：State、Memory、Artifact。

### 1.1 让会话活过进程重启

内存服务有个致命限制：**进程一退，对话就没了**。

把实现换成 `DatabaseSessionService`，指向一个 SQLite 文件，就能跨进程保留。下面这段不需要调用模型，只验证"写进去、换个服务实例还能读回来"：

In [1]:
import os
os.environ["LITELLM_LOCAL_MODEL_COST_MAP"] = "True"

from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.tools import ToolContext
from google.genai import types

APP, USER = "adk_ch06", "student"
session_service = InMemorySessionService()

async def run(root_agent, query, session_id, initial_state=None, show_state=True):
    existing = await session_service.get_session(app_name=APP, user_id=USER, session_id=session_id)
    if existing is None:
        await session_service.create_session(app_name=APP, user_id=USER,
                                             session_id=session_id, state=initial_state or {})
    runner = Runner(agent=root_agent, app_name=APP, session_service=session_service)
    msg = types.Content(role="user", parts=[types.Part(text=query)])
    print(f"🧑 {query}")
    async for ev in runner.run_async(user_id=USER, session_id=session_id, new_message=msg):
        if ev.is_final_response() and ev.content and ev.content.parts:
            print(f"🤖 {ev.content.parts[0].text}")
    session = await session_service.get_session(app_name=APP, user_id=USER, session_id=session_id)
    if show_state:
        print(f"🗂️ 当前 state → {dict(session.state)}\n" + "─" * 50)
    return session

print("✅ 环境就绪")


✅ 环境就绪


In [2]:
import os
import tempfile

from google.adk.sessions import DatabaseSessionService

db_path = os.path.join(tempfile.mkdtemp(), "adk_sessions.db")
db_url = f"sqlite+aiosqlite:///{db_path}"

# 第一个服务实例：写入一个会话
service_a = DatabaseSessionService(db_url=db_url)
created = await service_a.create_session(
    app_name=APP,
    user_id=USER,
    session_id="persist_demo",
    state={"user:pref_饮品": "乌龙茶"},
)
print("写入：", created.id, dict(created.state))

# 第二个服务实例：模拟"另一个进程"重新打开同一个数据库
service_b = DatabaseSessionService(db_url=db_url)
restored = await service_b.get_session(
    app_name=APP, user_id=USER, session_id="persist_demo"
)

assert restored is not None
assert restored.state["user:pref_饮品"] == "乌龙茶"
print("读回：", dict(restored.state))
print("✅ 会话已跨服务实例保留")

写入： persist_demo {'user:pref_饮品': '乌龙茶'}
读回： {'user:pref_饮品': '乌龙茶'}
✅ 会话已跨服务实例保留



### 1.2 列出、读取与删除会话

会话不是只能"创建后一直用"。`SessionService` 提供了管理接口：

| 操作 | 方法 | 说明 |
|---|---|---|
| 列出 | `list_sessions(app_name, user_id)` | 返回该用户的会话列表 |
| 读取 | `get_session(...)` | 不存在时返回 `None` |
| 删除 | `delete_session(...)` | 存储层删除，不可恢复 |

下面用内存服务验证这三件事，不需要模型：


In [3]:
lifecycle = InMemorySessionService()
await lifecycle.create_session(app_name=APP, user_id=USER, session_id="s1")
await lifecycle.create_session(app_name=APP, user_id=USER, session_id="s2")

listing = await lifecycle.list_sessions(app_name=APP, user_id=USER)
print("会话列表：", [s.id for s in listing.sessions])
assert {"s1", "s2"} <= {s.id for s in listing.sessions}

await lifecycle.delete_session(app_name=APP, user_id=USER, session_id="s1")
assert await lifecycle.get_session(app_name=APP, user_id=USER, session_id="s1") is None

listing = await lifecycle.list_sessions(app_name=APP, user_id=USER)
assert "s1" not in {s.id for s in listing.sessions}
print("✅ 列出、删除、删除后读取为 None，全部符合预期")

会话列表： ['s1', 's2']
✅ 列出、删除、删除后读取为 None，全部符合预期


---

## 2. State：带作用域前缀的工作记忆

`state` 是会话中的键值对，键名的前缀决定共享范围；存储后端决定保存时长：

| 前缀 | 作用域 | 谁可读取 | 典型用途 |
|---|---|---|---|
| （无前缀） | Session | 仅当前会话 | 本次任务的中间产物 |
| `user:` | 用户级 | 同一 app、同一 user 的会话 | 用户偏好、画像 |
| `app:` | 应用级 | 该应用下所有用户共享 | 全局配置 |
| `temp:` | 临时 | 仅当前这一轮，不落盘 | 中间计算 |

写入方式有两种：`create_session(state=...)` 初始化，或 `tool_context.state[...] = ...` 在工具里写。

运行时通过 State 接口的顶层赋值会记录 `state_delta`，随后随事件提交。嵌套对象原地修改不会自动记录该增量，练习 c4_2 会对比。第 5 章当前 Workflow 主线主要通过返回值传数据；旧 Agent 工作流也可通过 State 接力。

### 2.1 用 `user:` 前缀实现跨会话记忆

下面的管家会把偏好写入 `user:` 作用域。随后我们**换一个全新会话**，验证它依然生效。

**先猜：换一个 `session_id` 后，管家还记得"乌龙茶"吗？它记得的是"这个用户的偏好"，还是"上一个会话的上下文"？**

In [4]:
def remember_preference(item: str, value: str, tool_context: ToolContext) -> dict:
    """记录同一用户跨会话共享的偏好；保存时长取决于后端。

    Args:
        item: 偏好项目，例如 "饮品"、"音乐"。
        value: 偏好内容，例如 "乌龙茶"。
    """
    tool_context.state[f"user:pref_{item}"] = value   # user: 前缀 → 跨会话
    return {"status": f"已记住你对「{item}」的偏好"}

def recall_preferences(tool_context: ToolContext) -> dict:
    """查看当前已记住的所有用户偏好。"""
    prefs = {k: v for k, v in tool_context.state.to_dict().items() if k.startswith("user:pref_")}
    return {"preferences": prefs or "（还没有记录）"}

butler = Agent(
    name="butler", model=LiteLlm(model="deepseek/deepseek-chat"),
    instruction="你是贴身管家：用户提到喜好就调用 remember_preference；问偏好就调用 recall_preferences；回答时优先参考已知偏好。",
    description="跨会话记住用户偏好的管家",
    tools=[remember_preference, recall_preferences],
)

# 会话 A：告诉管家偏好
await run(butler, "记住：我最喜欢的饮品是乌龙茶。", session_id="session_A")


🧑 记住：我最喜欢的饮品是乌龙茶。


D:\Python\Lib\site-packages\google\adk\models\llm_request.py:273: UserWarning: [EXPERIMENTAL] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL is enabled.
  declaration = tool._get_declaration()


🤖 已经帮您记下了：您最喜欢的饮品是**乌龙茶**。以后不管是点单、选茶还是备茶点，我都会优先按乌龙茶来考虑。还有其他喜好想让我记住吗？
🗂️ 当前 state → {'user:pref_饮品': '乌龙茶'}
──────────────────────────────────────────────────


Session(id='session_A', app_name='adk_ch06', user_id='student', state={'user:pref_饮品': '乌龙茶'}, events=[Event(model_version=None, content=Content(
  parts=[
    Part(
      text='记住：我最喜欢的饮品是乌龙茶。'
    ),
  ],
  role='user'
), grounding_metadata=None, partial=None, turn_complete=None, turn_complete_reason=None, finish_reason=None, error_code=None, error_message=None, interrupted=None, custom_metadata=None, usage_metadata=None, live_session_resumption_update=None, live_session_id=None, go_away=None, voice_activity=None, input_transcription=None, output_transcription=None, avg_logprobs=None, logprobs_result=None, cache_metadata=None, citation_metadata=None, interaction_id=None, environment_id=None, invocation_id='e-ad4d4b58-06cf-4757-8d6b-5c7bb2e0b43d', author='user', actions=EventActions(skip_summarization=None, state_delta={}, artifact_delta={}, transfer_to_agent=None, escalate=None, requested_auth_configs={}, requested_tool_confirmations={}, compaction=None, end_of_agent=None, agent_stat

In [5]:
# 会话 B：全新 session_id，但 user: 前缀的数据应该还在
await run(butler, "我们以前聊过吗？你知道我喜欢喝什么吗？", session_id="session_B")


🧑 我们以前聊过吗？你知道我喜欢喝什么吗？
🤖 我们之前的会话记录我这里不能完整回看，但我确实记着您的偏好：

- **饮品：乌龙茶** 🍵

所以您喜欢喝乌龙茶，这点我记住了。要不要我顺手记点别的偏好，比如咖啡、口味或者作息？这样下次就不用您再重复了。
🗂️ 当前 state → {'user:pref_饮品': '乌龙茶'}
──────────────────────────────────────────────────


Session(id='session_B', app_name='adk_ch06', user_id='student', state={'user:pref_饮品': '乌龙茶'}, events=[Event(model_version=None, content=Content(
  parts=[
    Part(
      text='我们以前聊过吗？你知道我喜欢喝什么吗？'
    ),
  ],
  role='user'
), grounding_metadata=None, partial=None, turn_complete=None, turn_complete_reason=None, finish_reason=None, error_code=None, error_message=None, interrupted=None, custom_metadata=None, usage_metadata=None, live_session_resumption_update=None, live_session_id=None, go_away=None, voice_activity=None, input_transcription=None, output_transcription=None, avg_logprobs=None, logprobs_result=None, cache_metadata=None, citation_metadata=None, interaction_id=None, environment_id=None, invocation_id='e-7d81e378-ccf0-472d-96eb-66541158b812', author='user', actions=EventActions(skip_summarization=None, state_delta={}, artifact_delta={}, transfer_to_agent=None, escalate=None, requested_auth_configs={}, requested_tool_confirmations={}, compaction=None, end_of_agent=None, agent_

### 2.2 状态何时真正提交

第 3 章第 0 节已经验证了两端：直接改 `get_session()` 返回的副本无效；事件携带 `state_delta` 才会在下次读取时生效。这里补上中间那条——工具运行时拿到的 `State` 对象，以及它何时落库。

三种写法看起来都在"改 state"，区别在于**有没有一条事件承载增量**：

| 写法 | 是否提交 | 原因 |
|---|---|---|
| 取回 `session` 后直接 `session.state[k] = v` | ❌ | `Session.state` 是普通 dict，且拿到的是副本；没有事件 |
| `append_event` + `EventActions(state_delta={...})` | ✅ | 显式把增量写进事件，由会话服务合并 |
| 运行中 `tool_context.state[k] = v` | ✅ | `tool_context.state` 是带 pending delta 的 `State` 对象；工具返回时，增量随函数响应事件提交 |

因此 `tool_context.state` 的赋值不是"立刻落库"，而是**记在该次工具调用的增量里**。工具返回后事件被追加，State 才更新。

**先猜：工具里刚写完 `tool_context.state["draft"]`，紧接着再读它，能读到新值吗？** 能——`State` 会先查 pending delta；但这时值还没提交到会话存储，从服务端重新读取看不到。

In [6]:
from google.adk.events import Event, EventActions

commit_service = InMemorySessionService()
await commit_service.create_session(app_name=APP, user_id=USER, session_id="commit_demo")

session = await commit_service.get_session(
    app_name=APP, user_id=USER, session_id="commit_demo"
)
session.state["direct"] = "只在内存副本里"
again = await commit_service.get_session(
    app_name=APP, user_id=USER, session_id="commit_demo"
)
assert "direct" not in again.state, "直接改副本不应该落库"
print("直接修改副本后，重新读取：", dict(again.state))

event = Event(
    author="tool",
    content=types.Content(role="user", parts=[types.Part(text="更新状态")]),
    actions=EventActions(state_delta={"draft": "v1"}),
)
await commit_service.append_event(session, event)

committed = await commit_service.get_session(
    app_name=APP, user_id=USER, session_id="commit_demo"
)
assert committed.state["draft"] == "v1"
print("append_event 后：", dict(committed.state))

# tool_context.state 的本质：带 pending delta 的 State；赋值立即可读，但尚未提交
from google.adk.sessions.state import State

pending = State(value={"draft": "v0"}, delta={})
pending["draft"] = "v1"
assert pending["draft"] == "v1" and pending.has_delta()
print("✅ 状态提交必须经过事件，不能直接改副本")

直接修改副本后，重新读取： {}
append_event 后： {'draft': 'v1'}
✅ 状态提交必须经过事件，不能直接改副本


> 🔍 注意第二个会话打印的 state：`user:pref_饮品` 出现在了**全新会话**里。

框架在创建会话时自动把 `user:` / `app:` 作用域的数据合并进来。这就是 ADK 对"跨会话状态"的原生支持。

> ⚠️ **边界要分清**：`user:` 控制同一用户的共享范围，不决定保存时长。数据库 State 可以跨进程保留；内存 Memory 同样会随进程退出丢失。下一节改变的是读取方式：从精确键查询变成检索。

---

## 3. Memory：从键查询改为内容检索

State 按精确键读取；Memory 按查询检索候选内容。这里的区别是读取接口，不是“短期”和“永久”的区别。

```mermaid
flowchart LR
    S["会话结束"] --> ADD["memory_service<br/>.add_session_to_memory(session)"]
    ADD --> DB[("记忆库")]
    DB --> SEARCH["后续会话中<br/>search_memory(query)"]
    SEARCH --> CTX["注入上下文"]
    style DB fill:#e6f4ea,stroke:#34a853,stroke-width:2px
```

| 实现 | 检索方式 | 用途 |
|---|---|---|
| `InMemoryMemoryService` | 关键词匹配 | 学习原型 |
| `VertexAiMemoryBankService` | 语义检索（托管） | 需要云服务 |

本教程使用 `InMemoryMemoryService`。它不是语义检索，召回能力有限——这正好让你看清"记忆服务"的边界，而不是误以为它能理解一切。

工具或回调里也可以直接 `await tool_context.search_memory(query)`，它走同一个 `memory_service`，不需要再手工传入 `app_name` / `user_id`。

完整闭环：会话 1 沉淀记忆 → 会话 2 召回。

In [7]:
from google.adk.memory import InMemoryMemoryService
from google.adk.events import Event

memory_service = InMemoryMemoryService()
empty_memory = InMemoryMemoryService()
archive_service = InMemorySessionService()
archive = await archive_service.create_session(app_name=APP, user_id=USER, session_id="memory_source")
await archive_service.append_event(archive, Event(
    author="user",
    content=types.Content(role="user", parts=[types.Part(text="peanut allergy: 对花生过敏。")]),
))
archive = await archive_service.get_session(app_name=APP, user_id=USER, session_id="memory_source")
await memory_service.add_session_to_memory(archive)
found = await memory_service.search_memory(app_name=APP, user_id=USER, query="peanut")
empty = await empty_memory.search_memory(app_name=APP, user_id=USER, query="peanut")
other_user = await memory_service.search_memory(app_name=APP, user_id="another", query="peanut")
retrieved = [part.text for item in found.memories for part in item.content.parts or [] if part.text]
assert any("花生" in text for text in retrieved)
assert empty.memories == [] and other_user.memories == []
print({"实际召回": retrieved, "空服务": empty.memories, "其他用户": other_user.memories})

{'实际召回': ['peanut allergy: 对花生过敏。'], '空服务': [], '其他用户': []}


**证据与反例**：同一查询在已写入服务中返回原文，在空服务和另一用户下返回空列表。这里直接验证召回机制，没有用模型恰好避开某种食物反推成功。

**迁移**：给 Agent 装配 `PreloadMemoryTool` 并把这个 `memory_service` 传给 Runner，可以自动预加载检索结果；仍应记录实际召回，再评估答案是否使用它。关键词服务不理解否定或同义词，不应承担自动更新用户事实的职责。

> 🔍 本实验的证据是服务返回的记忆原文及空服务对照。最终回答只能作为另一层效果评估。

> ⚠️ **两个常见误解**：
>
> 1. **Memory 不是自动的。** 默认不会把每个会话都存进去，你需要显式调用 `add_session_to_memory`；
> 2. **关键词召回不等于语义理解。** `InMemoryMemoryService` 对同义词、否定句和信息更新都不敏感。生产系统要换语义检索，或自己维护结构化事实。

---

## 4. Artifact：管理"文件级"内容

State 适合存**小数据**：字符串、数字、小 JSON。

当 Agent 需要处理**文件**（报告、图片、PDF）时，用 `ArtifactService`。它有两个关键能力：

- **按会话隔离**：不同会话的文件不会混在一起；
- **版本管理**：同名文件多次保存，自动累加版本号。

| 实现 | 存储位置 | 用途 |
|---|---|---|
| `InMemoryArtifactService` | 内存 | 学习原型 |
| `FileArtifactService` | 本地目录 | 本地持久化 |

下面做一次完整实验：保存两版报告，列出所有版本，再读回第一版。

In [8]:
import tempfile

from google.adk.artifacts import FileArtifactService
from google.genai import types

root = tempfile.mkdtemp(prefix="adk_artifacts_")
artifacts = FileArtifactService(root_dir=root)

async def save(text: str) -> int:
    return await artifacts.save_artifact(
        app_name=APP, user_id=USER, session_id="report_demo",
        filename="report.md",
        artifact=types.Part.from_text(text=text),
    )

v0 = await save("# 报告 v1\n第一版内容")
v1 = await save("# 报告 v2\n第二版内容")

versions = await artifacts.list_artifact_versions(
    app_name=APP, user_id=USER, session_id="report_demo", filename="report.md"
)
print("版本号：", [v.version for v in versions])

old = await artifacts.load_artifact(
    app_name=APP, user_id=USER, session_id="report_demo",
    filename="report.md", version=v0,
)
print("读回第一版：", old.text)

assert v0 == 0 and v1 == 1
assert old.text.startswith("# 报告 v1")
print("✅ 版本管理可用")

版本号： [0, 1]
读回第一版： # 报告 v1
第一版内容
✅ 版本管理可用


---

## 5. 三层体系怎么选

| 你要存什么 | 用哪层 | 写法 |
|---|---|---|
| 本次任务的中间产物 | Session State（无前缀） | `state["draft"]` |
| 用户偏好（同一用户跨会话） | State `user:` 前缀 | `state["user:theme"]` |
| 全局配置 | State `app:` 前缀 | `state["app:version"]` |
| 一整轮对话的可检索沉淀 | Memory | `add_session_to_memory` / `search_memory` |
| 文件、大文本、需版本管理 | Artifact | `save_artifact` / `load_artifact` |

**一个判断顺序**：

1. **读取方式**：已知键且要求精确取值用 State；按问题检索历史用 Memory；文件用 Artifact。
2. **共享范围**：State 再选择 Session、同一用户、同一应用或本轮临时范围。
3. **保存时长**：独立选择内存或持久化后端，并设计删除、迁移和备份。
4. 无前缀 State 也能持久化；Memory 并不自动意味着永久保存。

> 📌 **别把 Artifact 当 State 用。** 更新整篇报告的 State 键会产生大增量；把它插入模型输入也会增加上下文开销；State 里只放文件名、版本号和摘要。

---

<details>
<summary><b>深入：与 LangGraph 对照</b></summary>

| ADK | LangGraph | 差异 |
|---|---|---|
| Session（events + state） | Thread（Checkpointer 持久化） | 高度对应 |
| `state` + 作用域前缀 | State 字段 | ADK 的前缀是内建语义 |
| `MemoryService` | Store（跨线程长期记忆） | 概念一致 |
| `ArtifactService` | 无直接对应 | ADK 特有 |
| `PreloadMemoryTool` | 节点里调用 store.search() | ADK 自动化程度更高 |

</details>

---

## 📌 用五句话复述本章

1. **Session 是一次对话的档案**，由 `app_name + user_id + session_id` 定位。
2. **State 的前缀决定共享范围**：无前缀=当前会话，`user:`=跨会话，`app:`=同一应用内共享，`temp:`=当前轮。
3. **Memory 需要显式沉淀**，`InMemoryMemoryService` 是关键词匹配，不是语义理解。
4. **Artifact 存文件并自动版本化**，大内容不要塞进 State。
5. **持久化只换实现**：`InMemorySessionService` → `DatabaseSessionService(db_url=...)`，业务流程不变。

> ➡️ 下一章：[07-回调与可观测性](07-回调与可观测性.ipynb) —— 把 Agent 打磨成生产级系统。

---

## 🧪 本章练习

### 1. 状态作用域实验（基础）

- [ ] 分别写入普通键、`user:` 键、`app:` 键和 `temp:` 键
- [ ] 建立两个用户、每个用户两个 Session
- [ ] 逐项验证哪些值共享、哪些值隔离
- [ ] 整理成"写入位置—可见范围—适用数据—不适用数据"表格

### 2. 跨会话用户偏好（进阶）

构建餐厅推荐助手。

- [ ] 在一个 Session 记录饮食偏好和过敏原
- [ ] 在新 Session 中仍能使用这些信息
- [ ] 另一用户不能读取该偏好
- [ ] 支持"忘记我的偏好"，并说明删除、覆盖、过期分别该由哪一层实现

### 3. 完成记忆闭环并评估召回（进阶）

- [ ] 准备五段含相似关键词但语义不同的历史对话
- [ ] 依次完成 Session 沉淀、查询/预加载、基于召回回答
- [ ] 为每个问题标注期望记忆，记录召回是否相关
- [ ] 分析关键词记忆在同义词、否定句、信息更新上的局限

### 4. Artifact 版本化报告（工程）

- [ ] 让 Agent 生成报告并保存为 Artifact
- [ ] 根据反馈生成第二版
- [ ] 支持列出版本和读取指定版本
- [ ] State 中只保存文件名、版本号与摘要
- [ ] 同一 Session 能准确引用两个版本

### 5. 设计一份记忆治理策略（开放）

针对企业助理：

- [ ] 规定哪些信息允许进入 Session、State、Memory、Artifact
- [ ] 补充保留期限、用户删除权、敏感信息脱敏、租户隔离与审计
- [ ] 映射到 LangGraph 的 Checkpointer 与 Store
- [ ] 指出两边最容易产生错误类比的地方

先做 [State 学生题](practice/student_state/README.md)：补全 TODO 后运行独立验收。验收使用真实 ToolContext，并检查未提交修改、嵌套修改和事件提交三个边界。
